# Task progress and fresh-pair rewards

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import math
class TaskReward:
    def __init__(self):self.reset()
    def reset(self):self.high_water_mark=0.0
    def calculate(self,raw_score):
        if not math.isfinite(raw_score):raise ValueError('Nonfinite raw score')
        before=self.high_water_mark;self.high_water_mark=max(before,float(raw_score))
        return float(self.high_water_mark>before)
class MaxSupportReward:
    def __init__(self):self.reset()
    def reset(self):self.seen_pair_ids=set()
    def calculate(self,q,valid,fresh,pair_id):
        if not valid or not fresh:return 0.0
        if q is None or not math.isfinite(q) or not 0<=q<=1:raise ValueError('Invalid fresh q')
        if pair_id is None:raise ValueError('Fresh pair identity required')
        key=tuple(pair_id)
        if key in self.seen_pair_ids:return 0.0
        self.seen_pair_ids.add(key);return float(q)
print('Task progress and fresh-pair rewards definitions/execution completed.')


Task progress and fresh-pair rewards definitions/execution completed.


In [3]:
task=TaskReward(); print('TASK:',[task.calculate(x) for x in [0,1,1,2]])
maximum=MaxSupportReward(); print('MAX first/duplicate:',maximum.calculate(.6,True,True,[1,1,2]),maximum.calculate(.6,True,True,[1,1,2]))

TASK: [0.0, 1.0, 0.0, 1.0]
MAX first/duplicate: 0.6 0.0
